## We will be using RoBERTa algorithm from Hugging Face library of transformers for NLP. Transformers are cutting edge technology now in NLP.

In [1]:
import os
import sys

sys.path.append("/Users/sakdag/Desktop/masters/weakly-supervised-fake-news-classification")

from src.utils.path_utils import get_project_root

fake_path = os.path.join(
    get_project_root(), "data/example/Fake.csv"
)
real_path = os.path.join(
    get_project_root(), "data/example/True.csv"
)
print(fake_path)
print(real_path)

/Users/sakdag/Desktop/masters/weakly-supervised-fake-news-classification/data/example/Fake.csv
/Users/sakdag/Desktop/masters/weakly-supervised-fake-news-classification/data/example/True.csv


In [2]:
import pandas as pd
import numpy as np

true, fake = pd.read_csv(real_path), pd.read_csv(fake_path)
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [3]:
fake['target'] = 0 
true['target'] = 1

In [4]:
fake

,title,text,subject,date,target
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0
...,...,...,...,...,...
23476,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016",0
23477,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016",0
23478,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016",0
23479,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016",0


In [5]:
news = pd.concat([true,fake],axis=0,ignore_index=True)
news = news.sample(frac = 1).reset_index(drop = True)
news.head()

,title,text,subject,date,target
0,BOOM! CLOCK BOY’S DAD LOSES Defamation Case In...,It s hard to be famous for being a victim in...,left-news,"Jan 10, 2017",0
1,"Clinton names close confidants, Obama veterans...",WASHINGTON/PHILADELPHIA (Reuters) - Hillary Cl...,politicsNews,"August 16, 2016",1
2,Ruby Rose And Chrissy Teigan Troll Donald Tru...,The longer Donald Trump and his merry band of ...,News,"March 15, 2017",0
3,WOW! FORMER MUSLIM’S MESSAGE To Refugees Who B...,"// <![CDATA[ (function(d, s, id) { var js, fjs...",politics,"Nov 25, 2015",0
4,Alec Baldwin Returns To SNL To Blast Trump’s ...,After what seemed like an agonizingly long wai...,News,"February 5, 2017",0


# Defining the tokenizer function

In [6]:
import tensorflow as tf
import transformers
import tqdm
from keras.preprocessing import sequence

#creating a function
def func_tokenizer(tokenizer_name, docs):
    features = []
    for doc in tqdm.tqdm(docs, desc = 'converting documents to features'):
        tokens = tokenizer_name.tokenize(doc)
        ids = tokenizer_name.convert_tokens_to_ids(tokens)
        features.append(ids)
    return features
print("The function is created successfully")

The function is created successfully


In [7]:
from sklearn.model_selection import train_test_split
X, y = news['text'], news['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# All hail Roberta from hugging face library of transformers 🤗

In [8]:
#Initialize bert tokenizer
roberta_tokenizer = transformers.RobertaTokenizer.from_pretrained('roberta-base-openai-detector')

In [9]:
roberta_train_features = func_tokenizer(roberta_tokenizer, X_train)
roberta_test_features = func_tokenizer(roberta_tokenizer, X_test)

converting documents to features: 100%|██████████| 8980/8980 [00:12<00:00, 693.47it/s]


In [10]:
roberta_trg = sequence.pad_sequences(roberta_train_features, maxlen = 500)
roberta_test = sequence.pad_sequences(roberta_test_features, maxlen = 500)

# XGBoost

In [11]:
import warnings
warnings.filterwarnings(action='ignore', category=UserWarning)
from xgboost import XGBClassifier
xgb = XGBClassifier(n_estimators = 1000, learning_rate = 0.15, max_depth = 9,
                    eval_metric = 'auc', use_label_encoder=False,objective = 'binary:logistic')
xgb.fit(roberta_trg, y_train)

XGBClassifier(base_score=0.5, booster='gbtree', callbacks=None,
              colsample_bylevel=1, colsample_bynode=1, colsample_bytree=1,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='auc', feature_types=None, gamma=0, gpu_id=-1,
              grow_policy='depthwise', importance_type=None,
              interaction_constraints='', learning_rate=0.15, max_bin=256,
              max_cat_threshold=64, max_cat_to_onehot=4, max_delta_step=0,
              max_depth=9, max_leaves=0, min_child_weight=1, missing=nan,
              monotone_constraints='()', n_estimators=1000, n_jobs=0,
              num_parallel_tree=1, predictor='auto', random_state=0, ...)

In [13]:
from sklearn.metrics import accuracy_score, roc_auc_score

xgb_pred = xgb.predict(roberta_test)
xgb_score = accuracy_score(y_test, xgb_pred)
xgb_roc = roc_auc_score(y_test, xgb_pred)
print("The accuracy of XGBOOST is: %0.2f" %xgb_score)
print("The roc_auc score of XGBOOST is: %0.2f" %xgb_roc)

The accuracy of XGBOOST is: 0.98
The roc_auc score of XGBOOST is: 0.98


In [14]:
from sklearn.metrics import classification_report
cr = classification_report(y_test, xgb_pred)
print(cr)

              precision    recall  f1-score   support

           0       0.99      0.98      0.98      4611
           1       0.98      0.99      0.98      4369

    accuracy                           0.98      8980
   macro avg       0.98      0.98      0.98      8980
weighted avg       0.98      0.98      0.98      8980



# CatBoost

In [13]:
from catboost import CatBoostClassifier
cb = CatBoostClassifier(eval_metric = 'Accuracy', iterations = 2000, learning_rate = 0.2)

In [14]:
cb.fit(roberta_trg, y_train, verbose = 0)
cb_pred = cb.predict(roberta_test)
cb_score = accuracy_score(y_test, cb_pred)
cb_roc = roc_auc_score(y_test, cb_pred)
print("The accuracy of CatBoost is: %0.2f" %cb_score)
print("The roc_auc score of CatBoost is: %0.2f" %cb_roc)

The accuracy of CatBoost is: 0.98
The roc_auc score of CatBoost is: 0.98


In [15]:
from sklearn.metrics import classification_report
cr2 = classification_report(y_test, cb_pred)
print(cr2)

              precision    recall  f1-score   support

           0       0.99      0.98      0.98      4676
           1       0.97      0.99      0.98      4304

    accuracy                           0.98      8980
   macro avg       0.98      0.98      0.98      8980
weighted avg       0.98      0.98      0.98      8980



# Upvote if you like it or fork it. This helps us motivate to produce more notebooks for the community.